# Notebook 03: Mask Fine-Tuning with Classification Heads

In this notebook, we perform end-to-end training but **freeze the entire Moirai encoder except for the mask tokens**. 

In [1]:
import torch
import pandas as pd
from IPython.display import display

from heads import (
    MeanPoolingClassifier,
    SingleScaleAttentionClassifier,
    SingleScaleMultiHeadClassifier,
    HierarchicalMultiHeadClassifier,
)
from models import MaskOnlyFinetunerWrapper, HeadFinetunerWrapper
from utils import get_lsst_dataloaders, universal_grid_search

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid = [1e-3]
wd_grid = [0.01, 0.05]

BATCH_SIZE = 256

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from utils import set_seed
set_seed(42)
print("Seed 42 — experiments are reproducible.")

Seed 42 — experiments are reproducible.


In [3]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [4]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 1. Baseline: Linear Model on Mask Embedding Only

In [5]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    print(f"Patch {patch}")
    _, metrics = universal_grid_search(
        model_class=MaskOnlyFinetunerWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid
    )
    df_mask_only.loc["Mask Only", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mask Only"] = metrics

Patch 8
LR=0.001| WD=0.01


 Early stopping : 28
Val Loss: 1.3579
LR=0.001| WD=0.05
 Early stopping : 28
Val Loss: 1.3439
Patch 16
LR=0.001| WD=0.01
 Early stopping : 24
Val Loss: 1.2649
LR=0.001| WD=0.05
 Early stopping : 30
Val Loss: 1.2671
Patch 32
LR=0.001| WD=0.01
 Early stopping : 26
Val Loss: 1.3225
LR=0.001| WD=0.05
 Early stopping : 23
Val Loss: 1.3278
Patch 64
LR=0.001| WD=0.01
 Early stopping : 23
Val Loss: 1.2373
LR=0.001| WD=0.05
 Early stopping : 25
Val Loss: 1.2283


In [6]:
print("Mask Only - Accuracy")
display(df_mask_only.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mask Only - Accuracy


Patch Size,8,16,32,64
Mask Only,0.5702,0.5811,0.5848,0.5843



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5702,0.385,0.3182,0.3237,0.5162,0.5702,0.5231


## 2. Baseline: Mean Pooling on Full Sequence (Context + Mask)
We average all patches (including the unfrozen mask) before passing them to a linear classifier.

In [7]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=HeadFinetunerWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid
    )
    df_mean_pool.loc["Mean Pooling", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mean Pooling"] = metrics

LR=0.001| WD=0.01
 Early stopping : 57
Val Loss: 1.1522
LR=0.001| WD=0.05
 Early stopping : 59
Val Loss: 1.1541
LR=0.001| WD=0.01
 Early stopping : 51
Val Loss: 1.1257
LR=0.001| WD=0.05
 Early stopping : 47
Val Loss: 1.1265
LR=0.001| WD=0.01
 Early stopping : 39
Val Loss: 1.1974
LR=0.001| WD=0.05


 Early stopping : 38
Val Loss: 1.1962
LR=0.001| WD=0.01
 Early stopping : 33
Val Loss: 1.1336
LR=0.001| WD=0.05
 Early stopping : 35
Val Loss: 1.1304


In [8]:
print("Mean Pooling - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mean Pooling - Accuracy


Patch Size,8,16,32,64
Mean Pooling,0.6111,0.6245,0.6058,0.6249



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5702,0.3850,0.3182,0.3237,0.5162,0.5702,0.5231
Mean Pooling,0.6111,0.4399,0.3673,0.3772,0.5609,0.6111,0.5672


## 3. Advanced Pooling: Attention & MHA (Context + Mask)
We use Attention mechanisms to dynamically weight the patches. The network can now learn to pay attention to specific context patches AND the fine-tuned mask patch.

In [9]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 64

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, metrics_attn = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[("Attention (Base)", mode), patch] = metrics_attn["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics_attn

        # 2. Multi-Head Attention
        _, metrics_mha = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(f"MHA (Heads={NUM_HEADS})", mode), patch] = metrics_mha["Accuracy"]

LR=0.001| WD=0.01
 Early stopping : 32
Val Loss: 1.2085
LR=0.001| WD=0.05


 Early stopping : 32
Val Loss: 1.2081
LR=0.001| WD=0.01
 Early stopping : 16
Val Loss: 1.1397
LR=0.001| WD=0.05
 Early stopping : 12
Val Loss: 1.1506
LR=0.001| WD=0.01
 Early stopping : 30
Val Loss: 1.2065
LR=0.001| WD=0.05
 Early stopping : 33
Val Loss: 1.1768
LR=0.001| WD=0.01
 Early stopping : 15
Val Loss: 1.1495
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.1562
LR=0.001| WD=0.01
 Early stopping : 28
Val Loss: 1.1553
LR=0.001| WD=0.05
 Early stopping : 29
Val Loss: 1.1880
LR=0.001| WD=0.01
 Early stopping : 14
Val Loss: 1.1212
LR=0.001| WD=0.05
 Early stopping : 11
Val Loss: 1.1215
LR=0.001| WD=0.01
 Early stopping : 34
Val Loss: 1.1968
LR=0.001| WD=0.05
 Early stopping : 27
Val Loss: 1.1809
LR=0.001| WD=0.01
 Early stopping : 15
Val Loss: 1.0946
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1194
LR=0.001| WD=0.01
 Early stopping : 26
Val Loss: 1.2322
LR=0.001| WD=0.05
 Early stopping : 29
Val Loss: 1.2410
LR=0.001| WD=0.01
 Early stopping : 13
Val Loss: 1.1826
LR=0.001| 

In [10]:
print("Attention & MHA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Attention & MHA - Accuracy


Patch Size                                8       16      32      64
Model            Mode                                               
Attention (Base) shared_context       0.5941  0.6119  0.5933  0.6338
                 independent_context  0.6123  0.5994  0.5892  0.6314
MHA (Heads=64)   shared_context       0.6180  0.6257  0.5957  0.6160
                 independent_context  0.6058  0.6233  0.6014  0.6265


Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5702,0.3850,0.3182,0.3237,0.5162,0.5702,0.5231
Mean Pooling,0.6111,0.4399,0.3673,0.3772,0.5609,0.6111,0.5672
Attention (shared_context),0.5941,0.4156,0.3590,0.3649,0.5344,0.5941,0.5492
Attention (independent_context),0.6123,0.4444,0.3594,0.3717,0.5628,0.6123,0.5627


## 4. Ablation Study: Number of Attention Heads
Testing the impact of `num_heads` on the Single-Scale MHA model (with mask fine-tuning) for a fixed patch size of 16.

In [11]:
MODES = ["shared_context", "independent_context"]
HEADS_TO_TEST = [4, 8, 16, 32, 64, 128, 384]

df_heads_ablation8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation8.index.name = "Mode"
df_heads_ablation8.columns.name = "Num Heads (Patch 8)"

for PATCH in [8]:
    for mode in MODES:
        for heads in HEADS_TO_TEST:
            print(f'{mode} - {heads}')
            _, metrics = universal_grid_search(
                model_class=HeadFinetunerWrapper,
                model_kwargs={
                    "head_class": SingleScaleMultiHeadClassifier,
                    "head_kwargs": {
                        "num_vars": NUM_VARS,
                        "num_classes": num_classes,
                        "patch_mode": mode,
                        "num_heads": heads
                    },
                    "patch_size": PATCH,
                    "num_vars": NUM_VARS,
                    "size": SIZE,
                    "remove_last_patch": False
                },
                train_loader=tr_loader,
                val_loader=va_loader,
                test_loader=te_loader,
                device=DEVICE,
                wd_grid=wd_grid, lr_grid=lr_grid
            )
            df_heads_ablation8.loc[mode, heads] = metrics["Accuracy"]
            df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics

shared_context - 4
LR=0.001| WD=0.01


 Early stopping : 13
Val Loss: 1.1482
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1415
shared_context - 8
LR=0.001| WD=0.01
 Early stopping : 11
Val Loss: 1.1700
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1384
shared_context - 16
LR=0.001| WD=0.01
 Early stopping : 15
Val Loss: 1.1577
LR=0.001| WD=0.05
 Early stopping : 12
Val Loss: 1.1722
shared_context - 32
LR=0.001| WD=0.01
 Early stopping : 15
Val Loss: 1.1595
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.1599
shared_context - 64
LR=0.001| WD=0.01
 Early stopping : 15
Val Loss: 1.1335
LR=0.001| WD=0.05
 Early stopping : 14
Val Loss: 1.1454
shared_context - 128
LR=0.001| WD=0.01
 Early stopping : 14
Val Loss: 1.1258
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.1478
shared_context - 384
LR=0.001| WD=0.01
 Early stopping : 14
Val Loss: 1.1460
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.1410
independent_context - 4
LR=0.001| WD=0.01
 Early stopping : 16
Val Loss: 1.1513
LR=0.001| WD=0.05
 Early stopping

In [12]:
print("Ablation: Num Heads (Patch = 8) - Accuracy")
display(df_heads_ablation8.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Ablation: Num Heads (Patch = 8) - Accuracy


Num Heads (Patch 8),4,8,16,32,64,128,384
Mode,,,,,,,
shared_context,0.6071,0.6026,0.6180,0.6184,0.6083,0.6067,0.603
independent_context,0.6030,0.6046,0.6103,0.6062,0.6058,0.6075,0.616



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5702,0.3850,0.3182,0.3237,0.5162,0.5702,0.5231
Mean Pooling,0.6111,0.4399,0.3673,0.3772,0.5609,0.6111,0.5672
Attention (shared_context),0.5941,0.4156,0.3590,0.3649,0.5344,0.5941,0.5492
Attention (independent_context),0.6123,0.4444,0.3594,0.3717,0.5628,0.6123,0.5627
MHA-4 (shared_context),0.6071,0.4899,0.3801,0.3928,0.5532,0.6071,0.5571
MHA-8 (shared_context),0.6026,0.4483,0.3603,0.3687,0.5579,0.6026,0.5590
MHA-16 (shared_context),0.6180,0.4716,0.3808,0.3990,0.5701,0.6180,0.5728
MHA-32 (shared_context),0.6184,0.4525,0.3831,0.3941,0.5545,0.6184,0.5702
MHA-64 (shared_context),0.6083,0.4560,0.3761,0.3881,0.5537,0.6083,0.5648


# 5. Hierarchical

In [13]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_hier = universal_grid_search(
            model_class=HeadFinetunerWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = metrics_hier["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical ({mode})"] = metrics_hier

LR=0.001| WD=0.01
 Early stopping : 19
Val Loss: 1.2595
LR=0.001| WD=0.05
 Early stopping : 21
Val Loss: 1.2660
LR=0.001| WD=0.01
 Early stopping : 23
Val Loss: 1.2081
LR=0.001| WD=0.05
 Early stopping : 23
Val Loss: 1.2129
LR=0.001| WD=0.01
 Early stopping : 13
Val Loss: 1.2478
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.2558
LR=0.001| WD=0.01
 Early stopping : 12
Val Loss: 1.2583
LR=0.001| WD=0.05
 Early stopping : 16
Val Loss: 1.2304
LR=0.001| WD=0.01
 Early stopping : 15
Val Loss: 1.2715
LR=0.001| WD=0.05
 Early stopping : 15
Val Loss: 1.2738
LR=0.001| WD=0.01
 Early stopping : 17
Val Loss: 1.2830
LR=0.001| WD=0.05
 Early stopping : 17
Val Loss: 1.2876
LR=0.001| WD=0.01
 Early stopping : 14
Val Loss: 1.2662
LR=0.001| WD=0.05
 Early stopping : 13
Val Loss: 1.2465
LR=0.001| WD=0.01
 Early stopping : 16
Val Loss: 1.2762
LR=0.001| WD=0.05
 Early stopping : 11
Val Loss: 1.2830


In [14]:
print("Hierarchical MHA - Accuracy")
display(df_hierarchical.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Hierarchical MHA - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.5762,0.5762,0.5714,0.5791
independent_context,0.5994,0.5888,0.5710,0.5819



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.5702,0.3850,0.3182,0.3237,0.5162,0.5702,0.5231
Mean Pooling,0.6111,0.4399,0.3673,0.3772,0.5609,0.6111,0.5672
Attention (shared_context),0.5941,0.4156,0.3590,0.3649,0.5344,0.5941,0.5492
Attention (independent_context),0.6123,0.4444,0.3594,0.3717,0.5628,0.6123,0.5627
MHA-4 (shared_context),0.6071,0.4899,0.3801,0.3928,0.5532,0.6071,0.5571
MHA-8 (shared_context),0.6026,0.4483,0.3603,0.3687,0.5579,0.6026,0.5590
MHA-16 (shared_context),0.6180,0.4716,0.3808,0.3990,0.5701,0.6180,0.5728
MHA-32 (shared_context),0.6184,0.4525,0.3831,0.3941,0.5545,0.6184,0.5702
MHA-64 (shared_context),0.6083,0.4560,0.3761,0.3881,0.5537,0.6083,0.5648


In [15]:
# ── Export Results to CSV ─────────────────────────────────────────────────────
import os
import numpy as np

os.makedirs("results_csv", exist_ok=True)

FINETUNING = "03_MaskFT"
MHA_HEADS  = 64
MODE_LABEL = {'shared_context': 'shared', 'independent_context': 'indep'}
rows = []

for patch in PATCH_SIZES:
    # Mean Pooling (no mode)
    acc = float(df_mean_pool.loc['Mean Pooling', patch])
    if patch == 8:
        rec = float(df_patch_8_metrics.loc['Mean Pooling', 'Macro Recall'])
        f1  = float(df_patch_8_metrics.loc['Mean Pooling', 'Macro F1'])
    else:
        rec, f1 = float('nan'), float('nan')
    rows.append({'finetuning': FINETUNING, 'pooling': 'Mean', 'patch_size': patch,
                 'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Attention — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_adv_single.loc[('Attention (Base)', mode), patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Attention ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Attention ({mode})', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Attention-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # MHA-64 — both modes (only evaluated at patch=8 in the ablation)
    for mode, lbl in MODE_LABEL.items():
        if patch == 8:
            acc = float(df_heads_ablation8.loc[mode, MHA_HEADS])
            rec = float(df_patch_8_metrics.loc[f'MHA-{MHA_HEADS} ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'MHA-{MHA_HEADS} ({mode})', 'Macro F1'])
        else:
            acc = float(df_adv_single.loc[('MHA (Heads=64)', mode), patch])
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'MHA-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

    # Hierarchical — both modes
    for mode, lbl in MODE_LABEL.items():
        acc = float(df_hierarchical.loc[mode, patch])
        if patch == 8:
            rec = float(df_patch_8_metrics.loc[f'Hierarchical ({mode})', 'Macro Recall'])
            f1  = float(df_patch_8_metrics.loc[f'Hierarchical ({mode})', 'Macro F1'])
        else:
            rec, f1 = float('nan'), float('nan')
        rows.append({'finetuning': FINETUNING, 'pooling': f'Hierarchical-{lbl}', 'patch_size': patch,
                     'accuracy': acc, 'macro_recall': rec, 'macro_f1': f1})

df_nb03 = pd.DataFrame(rows)
df_nb03.to_csv("results_csv/nb03_results.csv", index=False)
print("Saved results_csv/nb03_results.csv")
display(df_nb03)

Saved results_csv/nb03_results.csv


,finetuning,pooling,patch_size,accuracy,macro_recall,macro_f1
0,03_MaskFT,Mean,8,0.611111,0.367349,0.377171
1,03_MaskFT,Attention-shared,8,0.594079,0.358984,0.364854
2,03_MaskFT,Attention-indep,8,0.612328,0.359361,0.371658
3,03_MaskFT,MHA-shared,8,0.608273,0.376129,0.388078
4,03_MaskFT,MHA-indep,8,0.605839,0.357502,0.364266
5,03_MaskFT,Hierarchical-shared,8,0.576237,0.333172,0.328219
6,03_MaskFT,Hierarchical-indep,8,0.599351,0.369060,0.351916
7,03_MaskFT,Mean,16,0.624493,NaN,NaN
8,03_MaskFT,Attention-shared,16,0.611922,NaN,NaN
9,03_MaskFT,Attention-indep,16,0.599351,NaN,NaN


In [16]:
df_patch_8_metrics.to_csv("results_csv/nb03_patch_8_metrics.csv", index=True)